# Q5 Appendix: Full-Metric Article Tables (Case-Study vs Standard Cross-Network)

This appendix reproduces the four article-ready Q5 tables in **tidy/long format**:
every table has a single set of the four evaluation metrics reported as raw means
over runs — **Accuracy**, **Precision**, **Recall**, and **Macro-F1** (`f1_score`)
in that order — and a `Setting` label column whose values are `Standard` and
`Case Study` (fixed order Standard then Case Study).

**Method (identical filters/grouping to `Q5/Q5_analysis.ipynb`):**
- Case study (`wandb_case_study.csv`): drop `cnn`; drop split
  `Random with same distribution`; keep only rows where the set is a genuine
  held-out test set (`test_set in test_sets.split(',')`); exclude the SetA/SetC
  same-environment leak across train/val/test.
- For every metric m, the case-study value per set S is coalesced with
  `combine_first` in priority order
  `case_study/{S}/{m}` -> `case_study/{S}/event_evaluation/{m}` ->
  `case_study/{S}/component_evaluation/{m}`.
- Standard (`wandb_export_final_hyperparameters.csv`): CROSS-network only
  (`train_set != test_set`), SetA<->SetC excluded, all four metrics retained.
- Both sides averaged over `run_no`. Category map: DL = {gru, rnn}, else Classical.

Statistical machinery (Wilcoxon / Mann-Whitney p, Cohen's d, U) is dropped, and
so is any between-condition difference (the former `Δ MacroF1 (CS−Std)` reference
column). Configuration/run counts are kept where the source table had them.

## Setup and data loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths ────────────────────────────────────────────────────────────────
CS_PATH  = Path('../data/wandb_case_study.csv')
STD_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR  = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

NETWORKS = ['SetA', 'SetB', 'SetC', 'SetD']
METRICS  = ['accuracy', 'precision', 'recall', 'f1_score']  # f1_score = macro-F1
# Pretty column labels for the four metrics
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}

DL_CS = {'gru', 'rnn'}
def get_category(m):
    return 'Deep Learning' if m in DL_CS else 'Classical ML'

CS_MODELS  = ['catboost', 'gru', 'lightgbm', 'logistic-regression',
              'random-forest', 'rnn', 'xgboost']
STD_MODELS = ['gru', 'knn', 'lightgbm', 'logistic-regression', 'lstm',
              'mlp', 'nn', 'random-forest', 'rnn', 'svm', 'xgboost']
COMMON_MODELS = sorted(set(CS_MODELS) & set(STD_MODELS))

# ── Same-environment exclusion (SetA & SetC = same physical network) ───────
SAME_ENV_SET = frozenset(('SetA', 'SetC'))

def _same_env(a, b):
    return frozenset((a, b)) == SAME_ENV_SET

def cs_same_env_leak(row):
    tr, va, te = row['train_set'], row['val_set'], row['test_set']
    return _same_env(tr, va) or _same_env(tr, te) or _same_env(va, te)

# Shared caption suffix
CAP_SUFFIX = (r'Values are means over runs of Accuracy / Precision / Recall / '
              r'Macro-F1; case study (disjoint-network) vs standard cross-network '
              r'transfer. SetA$\leftrightarrow$SetC same-environment combinations '
              r'are excluded.')

### Load case study (coalesce all four metrics) and standard cross-network

In [2]:
# ── Case study ─────────────────────────────────────────────────────────────
cs_raw = pd.read_csv(CS_PATH)
cs_raw = cs_raw[cs_raw['model'] != 'cnn'].copy()
cs_raw = cs_raw[cs_raw['split'] != 'Random with same distribution'].copy()

# For each set S and each metric m, coalesce plain -> event_evaluation ->
# component_evaluation via combine_first (same priority order as f1 in Q5,
# applied PER METRIC).
cs_frames = []
id_cols = ['model', 'task', 'split', 'enable_sequences', 'run_no',
           'train_set', 'val_set', 'test_sets']
for s in NETWORKS:
    part = cs_raw[id_cols].copy()
    part['test_set'] = s
    for m in METRICS:
        val = (cs_raw[f'case_study/{s}/{m}']
               .combine_first(cs_raw[f'case_study/{s}/event_evaluation/{m}'])
               .combine_first(cs_raw[f'case_study/{s}/component_evaluation/{m}']))
        part[m] = val.values
    cs_frames.append(part)
# Keep a row if it has at least the primary metric (f1_score) present.
cs_long = pd.concat(cs_frames, ignore_index=True).dropna(subset=['f1_score'])

# Keep only genuine held-out test sets for each run.
cs_long = cs_long[cs_long.apply(
    lambda r: r['test_set'] in str(r['test_sets']).split(','), axis=1)].copy()
# Exclude SetA/SetC same-environment co-occurrence across train/val/test.
cs_long = cs_long[~cs_long.apply(cs_same_env_leak, axis=1)].copy()

# ── Standard cross-network ─────────────────────────────────────────────────
std_raw = pd.read_csv(STD_PATH, index_col=0)
std_raw = std_raw[std_raw['split'] != 'Random with same distribution'].copy()
metric_cols = [c for c in std_raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']
std_long = std_raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='mkey', value_name='value')
std_long[['train_set', 'test_set', 'metric']] = std_long['mkey'].str.split('/', expand=True)
std_long = std_long.drop(columns='mkey')
# Cross-network only, keep the four metrics.
std_cross = std_long[
    (std_long['train_set'] != std_long['test_set']) &
    (std_long['metric'].isin(METRICS))].copy()
std_cross = std_cross[~std_cross.apply(
    lambda r: _same_env(r['train_set'], r['test_set']), axis=1)].copy()

print(f'Case study rows : {len(cs_long):,}')
print(f'Standard rows   : {len(std_cross):,}')
print(cs_long[METRICS].notna().sum().to_string())

Case study rows : 5,056
Standard rows   : 24,080
accuracy     5056
precision    5056
recall       5056
f1_score     5056


### Average over runs into wide per-metric means

In [3]:
# ── Case study: mean over run_no per config, one column per metric ─────────
CS_KEYS = ['model', 'task', 'split', 'enable_sequences',
           'train_set', 'val_set', 'test_set']
cs_avg = (cs_long.groupby(CS_KEYS, as_index=False)[METRICS].mean())
cs_avg['category'] = cs_avg['model'].map(get_category)

# ── Standard: pivot metric to columns, mean over run_no per config ─────────
STD_KEYS = ['model', 'task', 'split', 'enable_sequences', 'train_set', 'test_set']
std_wide = (std_cross
            .pivot_table(index=STD_KEYS + ['run_no'], columns='metric',
                         values='value', aggfunc='mean')
            .reset_index())
std_avg = std_wide.groupby(STD_KEYS, as_index=False)[METRICS].mean()

print(f'cs_avg configs  : {len(cs_avg):,}')
print(f'std_avg configs : {len(std_avg):,}')

# ── Tidy/long helpers ──────────────────────────────────────────────────────
# Metric columns are now a single, fixed-order block (no Std/CS prefixing).
METRIC_ORDER = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_NAMES = [METRIC_LABEL[m] for m in METRIC_ORDER]  # display column names
SETTING_ORDER = ['Standard', 'Case Study']              # fixed row order

def metric_values(row_src, keys):
    """Return {display_name: value} for the four metrics, in fixed order."""
    return {METRIC_LABEL[m]: row_src[k] for m, k in zip(METRIC_ORDER, keys)}

def latex_table(df, caption, label, path):
    """Write a LaTeX table (index dropped) with the repo's formatting."""
    tex = df.to_latex(index=False, float_format='%.4f', escape=False,
                      caption=caption, label=label)
    Path(path).write_text(tex)

cs_avg configs  : 574
std_avg configs : 680


## q5_appendix_overall

Source: `table1_overall_comparison`, tidy/long. Columns
`[Condition, Setting, N configs, Accuracy, Precision, Recall, Macro-F1]`.
Each Condition (Overall, per split, per sequences) emits two rows — `Standard`
then `Case Study`. Pairing matches on
`(model, task, split, enable_sequences, train_set, test_set)` (val_set kept on CS).

In [4]:
# Paired: match CS to Std on the shared config keys (val_set stays on CS side).
paired = cs_avg.merge(
    std_avg, on=['model', 'task', 'split', 'enable_sequences', 'train_set', 'test_set'],
    suffixes=('_cs', '_std'), how='inner')

def overall_rows(condition, sub):
    """Emit two tidy rows (Standard, Case Study) for a condition."""
    n = len(sub)
    std_rec = {'Condition': condition, 'Setting': 'Standard', 'N configs': n}
    std_rec.update(metric_values(
        {f'{m}_std': sub[f'{m}_std'].mean() for m in METRIC_ORDER},
        [f'{m}_std' for m in METRIC_ORDER]))
    cs_rec = {'Condition': condition, 'Setting': 'Case Study', 'N configs': n}
    cs_rec.update(metric_values(
        {f'{m}_cs': sub[f'{m}_cs'].mean() for m in METRIC_ORDER},
        [f'{m}_cs' for m in METRIC_ORDER]))
    return [std_rec, cs_rec]

records = []
records += overall_rows('Overall', paired)
for sp in sorted(paired['split'].unique()):
    records += overall_rows(f'Split: {sp}', paired[paired['split'] == sp])
for seq in [True, False]:
    records += overall_rows(f'Sequences: {seq}', paired[paired['enable_sequences'] == seq])

col_order = ['Condition', 'Setting', 'N configs'] + METRIC_NAMES
t1 = pd.DataFrame(records)[col_order]

t1.round(4).to_csv(TAB_DIR / 'q5_appendix_overall.csv', index=False)
latex_table(t1.round(4),
    r'Overall case-study (disjoint-network) vs standard cross-network performance, stratified by split strategy and feature approach; one row per setting. ' + CAP_SUFFIX,
    'tab:q5_appendix_overall', TAB_DIR / 'q5_appendix_overall.tex')
print('Saved q5_appendix_overall ', t1.shape)
t1.round(4)

Saved q5_appendix_overall  (10, 7)


,Condition,Setting,N configs,Accuracy,Precision,Recall,Macro-F1
0,Overall,Standard,531,0.7685,0.6380,0.6093,0.5545
1,Overall,Case Study,531,0.8508,0.6652,0.6458,0.5993
2,Split: Random split,Standard,281,0.7535,0.6310,0.6032,0.5523
3,Split: Random split,Case Study,281,0.8524,0.6667,0.6454,0.6007
4,Split: Time split,Standard,250,0.7854,0.6457,0.6162,0.5569
5,Split: Time split,Case Study,250,0.8491,0.6634,0.6463,0.5977
6,Sequences: True,Standard,325,0.8160,0.6581,0.6281,0.5597
7,Sequences: True,Case Study,325,0.8777,0.6775,0.6366,0.6009
8,Sequences: False,Standard,206,0.6937,0.6062,0.5798,0.5463
9,Sequences: False,Case Study,206,0.8084,0.6457,0.6604,0.5968


## q5_appendix_pairing

Source: `table2_pairing_ranking`, tidy/long. Columns
`[Train set, Val set, Setting, N runs, Accuracy, Precision, Recall, Macro-F1]`.
Each (Train set, Val set) pairing emits two rows — `Standard` then `Case Study`.
The standard side is the mean over all cross-network configs sharing that
training source. (Train, Val) groups are sorted by case-study Macro-F1
descending; the old Rank column is dropped.

In [5]:
# Standard per-train-set means (one value per metric per training source).
std_by_train = std_avg.groupby('train_set')[METRICS].mean()

records = []
for (tr, va), grp in cs_avg.groupby(['train_set', 'val_set']):
    n_runs = len(grp)
    cs_f1 = grp['f1_score'].mean()
    # Standard row: cross-network mean for the same training source.
    std_rec = {'Train set': tr, 'Val set': va, 'Setting': 'Standard',
               'N runs': n_runs, '_cs_f1': cs_f1}
    for m in METRIC_ORDER:
        std_rec[METRIC_LABEL[m]] = (std_by_train.loc[tr, m]
                                    if tr in std_by_train.index else float('nan'))
    # Case Study row: mean over this pairing's configs.
    cs_rec = {'Train set': tr, 'Val set': va, 'Setting': 'Case Study',
              'N runs': n_runs, '_cs_f1': cs_f1}
    for m in METRIC_ORDER:
        cs_rec[METRIC_LABEL[m]] = grp[m].mean()
    records.append(std_rec)
    records.append(cs_rec)

t2 = pd.DataFrame(records)
# Sort (Train, Val) groups by case-study Macro-F1 desc, then Setting fixed order.
t2['Setting'] = pd.Categorical(t2['Setting'], categories=SETTING_ORDER, ordered=True)
t2 = (t2.sort_values(['_cs_f1', 'Setting'], ascending=[False, True])
        .drop(columns='_cs_f1')
        .reset_index(drop=True))
col_order = ['Train set', 'Val set', 'Setting', 'N runs'] + METRIC_NAMES
t2 = t2[col_order]

t2.round(4).to_csv(TAB_DIR / 'q5_appendix_pairing.csv', index=False)
latex_table(t2.round(4),
    r'Case-study vs standard cross-network test metrics for each (train set, validation set) pairing, sorted by case-study Macro-F1; one row per setting. Standard rows are the cross-network mean for the same training source. ' + CAP_SUFFIX,
    'tab:q5_appendix_pairing', TAB_DIR / 'q5_appendix_pairing.tex')
print('Saved q5_appendix_pairing ', t2.shape)
t2.round(4)

Saved q5_appendix_pairing  (20, 8)


,Train set,Val set,Setting,N runs,Accuracy,Precision,Recall,Macro-F1
0,SetC,SetD,Standard,34,0.8349,0.7653,0.6825,0.6584
1,SetC,SetD,Case Study,34,0.9896,0.9033,0.9045,0.8938
2,SetB,SetC,Standard,39,0.7864,0.6485,0.5710,0.5441
3,SetB,SetC,Case Study,39,0.8831,0.8021,0.6620,0.6716
4,SetA,SetB,Standard,74,0.8527,0.6750,0.5584,0.5644
5,SetA,SetB,Case Study,74,0.8987,0.7728,0.6379,0.6603
6,SetC,SetB,Standard,33,0.8349,0.7653,0.6825,0.6584
7,SetC,SetB,Case Study,33,0.9281,0.8214,0.6106,0.6434
8,SetB,SetA,Standard,39,0.7864,0.6485,0.5710,0.5441
9,SetB,SetA,Case Study,39,0.8712,0.7792,0.6232,0.6234


## q5_appendix_per_model

Source: `table3_per_model`, tidy/long. Columns
`[model, category, Setting, Accuracy, Precision, Recall, Macro-F1]`. Each model
emits a `Case Study` row and — only if it also appears on the standard side
(common models) — a `Standard` row. Models present only in the case study
(catboost, and any sequence-only / classical models absent from the standard
cross-network set) simply omit their Standard row. Models are sorted by
case-study Macro-F1 descending, then Setting fixed order.

In [6]:
cs_m = cs_avg.groupby(['model', 'category'])[METRICS].mean().reset_index()
std_m = (std_avg[std_avg['model'].isin(COMMON_MODELS)]
         .groupby('model')[METRICS].mean())

records = []
for _, r in cs_m.iterrows():
    model, cat = r['model'], r['category']
    cs_f1 = r['f1_score']
    # Standard row only for models present on the standard side.
    if model in std_m.index:
        std_rec = {'model': model, 'category': cat, 'Setting': 'Standard',
                   '_cs_f1': cs_f1}
        for m in METRIC_ORDER:
            std_rec[METRIC_LABEL[m]] = std_m.loc[model, m]
        records.append(std_rec)
    # Case Study row for every model.
    cs_rec = {'model': model, 'category': cat, 'Setting': 'Case Study',
              '_cs_f1': cs_f1}
    for m in METRIC_ORDER:
        cs_rec[METRIC_LABEL[m]] = r[m]
    records.append(cs_rec)

t3 = pd.DataFrame(records)
t3['Setting'] = pd.Categorical(t3['Setting'], categories=SETTING_ORDER, ordered=True)
t3 = (t3.sort_values(['_cs_f1', 'Setting'], ascending=[False, True])
        .drop(columns='_cs_f1')
        .reset_index(drop=True))
col_order = ['model', 'category', 'Setting'] + METRIC_NAMES
t3 = t3[col_order]

t3.round(4).to_csv(TAB_DIR / 'q5_appendix_per_model.csv', index=False)
latex_table(t3.round(4),
    r'Per-model case-study vs standard cross-network performance, sorted by case-study Macro-F1; one row per setting. Models absent from the standard cross-network evaluation (e.g. catboost) have no Standard row. ' + CAP_SUFFIX,
    'tab:q5_appendix_per_model', TAB_DIR / 'q5_appendix_per_model.tex')
print('Saved q5_appendix_per_model ', t3.shape)
t3.round(4)

Saved q5_appendix_per_model  (18, 7)


,model,category,Setting,Accuracy,Precision,Recall,Macro-F1
0,xgboost,Classical ML,Standard,0.7929,0.6671,0.6235,0.5897
1,xgboost,Classical ML,Case Study,0.8916,0.7125,0.6620,0.6474
2,logistic-regression,Classical ML,Standard,0.7973,0.6393,0.5710,0.5582
3,logistic-regression,Classical ML,Case Study,0.8882,0.6978,0.6391,0.6290
4,lightgbm,Classical ML,Standard,0.7950,0.6650,0.6295,0.5939
5,lightgbm,Classical ML,Case Study,0.8591,0.6765,0.6517,0.6115
6,random-forest,Classical ML,Standard,0.7872,0.6664,0.6364,0.5940
7,random-forest,Classical ML,Case Study,0.8019,0.6840,0.6553,0.5962
8,catboost,Classical ML,Case Study,0.8021,0.6532,0.6235,0.5943
9,mlp,Classical ML,Case Study,0.8743,0.6352,0.6315,0.5890


## q5_appendix_per_pairing_test

Source: `table4_statistical_tests` granularity — the stat machinery is replaced
with raw means, tidy/long. Columns
`[Train set, Val set, Setting, Accuracy, Precision, Recall, Macro-F1]`. Each
pairing emits two rows — `Standard` then `Case Study`. The case-study means are
over that pairing's configs; the standard directional means are over the same
training source restricted to the same test networks the pairing actually covers
(`std_avg[train_set==tr & test_set in test_nets]`), matching the source table's
`std_vals` selection. Source (Train, Val) order is preserved.

In [7]:
records = []
for (tr, va), grp in cs_avg.groupby(['train_set', 'val_set']):
    test_nets = grp['test_set'].unique()
    std_sel = std_avg[(std_avg['train_set'] == tr) &
                      (std_avg['test_set'].isin(test_nets))]
    # Standard row (directional means for this pairing's test networks).
    std_rec = {'Train set': tr, 'Val set': va, 'Setting': 'Standard'}
    for m in METRIC_ORDER:
        std_rec[METRIC_LABEL[m]] = std_sel[m].mean() if len(std_sel) else float('nan')
    # Case Study row.
    cs_rec = {'Train set': tr, 'Val set': va, 'Setting': 'Case Study'}
    for m in METRIC_ORDER:
        cs_rec[METRIC_LABEL[m]] = grp[m].mean()
    records.append(std_rec)
    records.append(cs_rec)

# Preserve source (Train, Val) groupby order (Standard then Case Study within each).
col_order = ['Train set', 'Val set', 'Setting'] + METRIC_NAMES
t4 = pd.DataFrame(records)[col_order]

t4.round(4).to_csv(TAB_DIR / 'q5_appendix_per_pairing_test.csv', index=False)
latex_table(t4.round(4),
    r'Per-(train set, validation set) directional means: case study vs standard cross-network for the same training source restricted to the pairing''s test networks; one row per setting. ' + CAP_SUFFIX,
    'tab:q5_appendix_per_pairing_test', TAB_DIR / 'q5_appendix_per_pairing_test.tex')
print('Saved q5_appendix_per_pairing_test ', t4.shape)
t4.round(4)

Saved q5_appendix_per_pairing_test  (20, 7)


,Train set,Val set,Setting,Accuracy,Precision,Recall,Macro-F1
0,SetA,SetB,Standard,0.8569,0.7739,0.6387,0.6464
1,SetA,SetB,Case Study,0.8987,0.7728,0.6379,0.6603
2,SetA,SetD,Standard,0.8485,0.5761,0.4781,0.4825
3,SetA,SetD,Case Study,0.8672,0.6020,0.5289,0.5363
4,SetB,SetA,Standard,0.7909,0.7418,0.6057,0.5747
5,SetB,SetA,Case Study,0.8712,0.7792,0.6232,0.6234
6,SetB,SetC,Standard,0.7909,0.7418,0.6057,0.5747
7,SetB,SetC,Case Study,0.8831,0.8021,0.6620,0.6716
8,SetB,SetD,Standard,0.7842,0.6018,0.5536,0.5289
9,SetB,SetD,Case Study,0.7735,0.6094,0.5747,0.5595
